# Findings

We looked at 79 BART and Caltrain stations across the Bay Area and measured how many essential amenities (grocery stores, parks, clinics, pharmacies, and 
childcare) are within walking distance of each one. Here is what we found.

In [15]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
import statsmodels.api as sm

DATA_PATH = "../data/processed/final_station_data.csv"
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

required = {
    "station_name": "Unknown",
    "agency": "—",
    "station_type": "unknown",
    "latitude": None,
    "longitude": None,
    "total_amenities": 0,
    "grocery": 0,
    "park": 0,
    "clinic": 0,
    "pharmacy": 0,
    "childcare": 0,
    "ridership": 0,
    "median_income": np.nan,
    "pct_no_vehicle": np.nan,
    "pct_nonwhite": np.nan,
    "unmet_need_index": np.nan,
    "amenity_entropy": np.nan,
}
for col, default in required.items():
    if col not in df.columns:
        df[col] = default

df = df.dropna(subset=["latitude", "longitude"])
df["station_type"] = df["station_type"].str.lower().fillna("unknown")

core = df[df["station_type"] == "core"]
peri = df[df["station_type"] == "peripheral"]

COLORS = {
    "core":       "#378ADD",
    "peripheral": "#D85A30",
    "unknown":    "#888780",
    "unmet_ring": "rgba(120, 3, 179, 0.33)"
}

from IPython.display import display, HTML
display(HTML("<script src='https://cdn.plot.ly/plotly-latest.min.js'></script>"))

## What the data shows

The charts below summarize the core-peripheral gap from three angles: overall amenity counts, which stations have the most urgent unmet need, and how the gap breaks down by amenity type. For the full statistical tests behind these patterns, see **Methods & Results**.



```{raw} html
<div class="chart-callout">
  <strong>What this shows.</strong> Figure 1 puts both groups side by side, and the gap is 
clear. Some core stations have 50+ amenities nearby. Others, like Coliseum and Bay Fair, have 3 and 5. 
</div>
```

In [18]:
import plotly.graph_objects as go
from IPython.display import display, HTML

fig_dist = go.Figure()

for stype, color, name in [
    ("core", "#378ADD", "Core"),
    ("peripheral", "#D85A30", "Peripheral"),
]:
    sub = df[df["station_type"] == stype]["total_amenities"].dropna()
    fig_dist.add_trace(go.Box(
        y=sub, name=name,
        marker_color=color,
        boxpoints="all", jitter=0.3, pointpos=-1.8,
        marker=dict(size=5, opacity=0.6),
    ))

fig_dist.update_layout(
    title="Total amenities: core vs peripheral",
    yaxis_title="Total amenities (½-mile radius)",
    height=420, width=560,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    showlegend=False,
    margin=dict(l=50, r=20, t=50, b=40),
)

display(HTML(fig_dist.to_html(
    full_html=False,
    include_plotlyjs="cdn",
)))

*Figure 1. Distribution of total walkable amenities (½-mile radius) by station type. Each point is one station. Data: OpenStreetMap amenity extracts, FY2025 BART and Caltrain ridership, n=79 stations.*



```{raw} html
<div class="chart-callout">
  <strong>What this shows.</strong> Figure 2 ranks stations by how much their car-free 
population exceeds what the neighborhood can offer on foot. Coliseum, Bay Fair, and MacArthur are all 
high-ridership core stations by the numbers, but they look peripheral when you 
count what's around them.
</div>
```

In [6]:
# ── 4b: Unmet need index — top 15 stations ───────────────────────────────────
top15 = df.nlargest(15, "unmet_need_index")[["station_name", "station_type", "unmet_need_index"]].copy()
top15["color"] = top15["station_type"].map(COLORS).fillna(COLORS["unknown"])
top15 = top15.sort_values("unmet_need_index")

fig_unmet = go.Figure(go.Bar(
    x=top15["unmet_need_index"],
    y=top15["station_name"],
    orientation="h",
    marker_color=top15["color"],
    text=top15["unmet_need_index"].round(3),
    textposition="outside",
))
fig_unmet.update_layout(
    title="Top 15 stations by unmet need index",
    xaxis_title="Unmet need index",
    height=480, width=640,
    paper_bgcolor="white", plot_bgcolor="white",
    xaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=160, r=60, t=50, b=40),
)
display(HTML(fig_unmet.to_html(full_html=False, include_plotlyjs=False)))

*Figure 2. Top 15 stations ranked by unmet need index. Blue = core, orange = peripheral. Index = % zero-vehicle households × (1 − normalized amenity count), min-max normalized across all 79 stations. Data: ACS 2024 5-year estimates, OpenStreetMap.*




```{raw} html
<div class="chart-callout">
  <strong>What this shows.</strong> Figure 3 breaks the gap down by category, and the clinic 
numbers are the ones that matter most for daily life. At peripheral stations, the 
average rider is within walking distance of less than one clinic. At core stations, 
that number is 2.5. Pharmacies follow the same pattern.
</div>
```



```{raw} html
<div class="chart-callout-red">
  <strong>Note on category selection.</strong> The chart below shows the five amenity categories with the largest and most consistent gaps after FDR correction. Convenience stores showed the largest raw effect (Glass Delta = 2.47) but were excluded here because convenience store counts are heavily skewed by a few downtown SF stations, making the mean misleading as a summary. The full 10-category breakdown is in Methods & Results.
</div>
```

In [7]:
# ── 4c: Amenity category breakdown — grouped bar ─────────────────────────────
cats   = ["grocery", "park", "clinic", "pharmacy", "childcare"]
labels = ["Grocery", "Parks", "Clinics", "Pharmacy", "Childcare"]
colors_bar = ["#378ADD", "#1D9E75", "#D85A30", "#7F77DD", "#EF9F27"]

core_means = [core[c].mean() for c in cats]
peri_means = [peri[c].mean() for c in cats]

fig_bar = go.Figure([
    go.Bar(name="Core",       x=labels, y=core_means, marker_color=COLORS["core"]),
    go.Bar(name="Peripheral", x=labels, y=peri_means, marker_color=COLORS["peripheral"]),
])
fig_bar.update_layout(
    barmode="group",
    title="Mean amenity count by category (core vs peripheral)",
    yaxis_title="Mean count (½-mile radius)",
    height=420, width=620,
    paper_bgcolor="white", plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=50, r=20, t=60, b=40),
)
display(HTML(fig_bar.to_html(full_html=False, include_plotlyjs=False)))

*Figure 3. Mean amenity count within ½ mile by category and station type. Hospitals, convenience stores, doctors, and kindergartens are excluded from this summary chart because their permutation test results were not significant after FDR correction. Data: OpenStreetMap, n=40 core, n=39 peripheral.*



```{raw} html
<div class="chart-callout">
  <strong>What this shows.</strong> Figure 4 maps out the relationship between car-free 
households and amenity access directly — having more car-free households nearby 
tends to mean more amenities, but that breaks down for several East Oakland 
stations. Coliseum sits at 29% car-free households with only 3 amenities nearby. 
The trendline goes up; those stations go sideways.

</div>
```




In [33]:
fig_scatter = px.scatter(
    df,
    x="pct_no_vehicle",
    y="total_amenities",
    color="station_type",
    color_discrete_map={
        "core": "#378ADD",
        "peripheral": "#D85A30",
    },
    hover_name="station_name",
    hover_data={
        "agency": True,
        "unmet_need_index": ":.3f",
        "station_type": False,
        "pct_no_vehicle": ":.1f",
        "total_amenities": True,
    },
    labels={
        "pct_no_vehicle": "% zero-vehicle households (tract level)",
        "total_amenities": "Total amenities (½-mile radius)",
        "station_type": "Station type",
        "unmet_need_index": "Unmet need index",
    },
    trendline="ols",
    trendline_scope="overall",
    trendline_color_override="#888",
    height=450,
    width=640,
)

fig_scatter.update_traces(
    marker=dict(size=8, opacity=0.75),
    selector=dict(mode="markers"),
)

fig_scatter.update_layout(
    paper_bgcolor="white",
    plot_bgcolor="white",
    yaxis=dict(gridcolor="#F1EFE8"),
    xaxis=dict(gridcolor="#F1EFE8"),
    margin=dict(l=50, r=20, t=30, b=50),
    legend=dict(
        title="Station type",
        x=0.01, y=0.99,
        bgcolor="rgba(255,255,255,0.88)",
        bordercolor="#D3D1C7",
        borderwidth=0.5,
    ),
)

outliers = df[df["unmet_need_index"] >= df["unmet_need_index"].quantile(0.9)]

label_these = ["Coliseum", "Pittsburg Center", "Bay Fair", "MacArthur", "Balboa Park"]


# Manual offsets to avoid overlap
offsets = {
    "Coliseum": (40, 20),
    "Pittsburg Center": (-5, 20),
    "Bay Fair": (0, 15),
    "MacArthur": (40, 20),
    "Balboa Park": (40, -25),
}

for _, row in df[df["station_name"].isin(label_these)].iterrows():
    ax, ay = offsets.get(row["station_name"], (40, -20))
    fig_scatter.add_annotation(
        x=row["pct_no_vehicle"],
        y=row["total_amenities"],
        text=row["station_name"],
        showarrow=True,
        arrowhead=2,
        arrowsize=0.8,
        arrowcolor="#aaa",
        font=dict(size=9, color="#444"),
        ax=ax, ay=ay,
    )

display(HTML(fig_scatter.to_html(
    full_html=False,
    include_plotlyjs="cdn",
)))

*Figure 4. % zero-vehicle households vs. total walkable amenities. Gray line = OLS trend (Spearman rho = +0.476, p < 0.001). Labeled stations are in the top decile of unmet need. Data: ACS 2024, OpenStreetMap.*

## Unmet need across the network

The map below shows unmet need index at each station, colored from light to dark. Darker = more urgent. Stations with a black outline are in the top quartile. These are the places where the combination of low amenity access and high car-free household rates is most severe. Use this alongside the **Station Access Explorer** map to get a sense of where the network is failing its most transit-dependent riders geographically.

In [20]:
import plotly.express as px
import pandas as pd
import numpy as np

# Build the choropleth as a scatter map colored by unmet need
fig_choro = px.scatter_mapbox(
    df,
    lat="latitude",
    lon="longitude",
    color="unmet_need_index",
    size="total_amenities",
    size_max=20,
    color_continuous_scale=[
    [0.0,  "#d4b9da"],
    [0.25, "#c994c7"],
    [0.5,  "#df65b0"],
    [0.75, "#ce1256"],
    [1.0,  "#67001f"],
    ],
    hover_name="station_name",
    hover_data={
        "agency": True,
        "station_type": True,
        "total_amenities": True,
        "unmet_need_index": ":.3f",
        "pct_no_vehicle": ":.1f",
        "latitude": False,
        "longitude": False,
    },
    labels={
        "unmet_need_index": "Unmet need index",
        "total_amenities": "Total amenities",
        "station_type": "Type",
        "pct_no_vehicle": "% no vehicle",
    },
    mapbox_style="carto-positron",
    center={"lat": df["latitude"].mean(), "lon": df["longitude"].mean()},
    zoom=8.5,
    height=520,
)

# Outline the top quartile stations
threshold = df["unmet_need_index"].quantile(0.75)
top_q = df[df["unmet_need_index"] >= threshold]

import plotly.graph_objects as go
fig_choro.add_trace(go.Scattermapbox(
    lat=top_q["latitude"],
    lon=top_q["longitude"],
    mode="markers",
    marker=dict(
        size=18,
        color="rgba(0,0,0,0)",
        opacity=1,
    ),
    hoverinfo="skip",
    showlegend=False,
))

fig_choro.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    paper_bgcolor="white",
    coloraxis_colorbar=dict(
        title="Unmet need",
        thickness=14,
        len=0.5,
        tickformat=".2f",
    ),
)

# fig_choro.show()

display(HTML(fig_choro.to_html(
    full_html=False,
    include_plotlyjs="cdn",
)))

## Three-way equity view

The figure below plots each station across three dimensions at once: car-free household rate, non-white resident share, and total amenities. Color shows unmet need index. 
Stations in the dark red corner (high on both demographic axes, low on amenities) are where the equity gap is most acute. Rotate the chart to explore the relationship from different angles.

In [37]:
fig_3d = px.scatter_3d(
    df,
    x="pct_no_vehicle",
    y="pct_nonwhite",
    z="total_amenities",
    color="unmet_need_index",
    color_continuous_scale=[
        [0.0, "#f7f4f9"],
        [0.5, "#df65b0"],
        [1.0, "#67001f"],
    ],
    hover_name="station_name",
    hover_data={
        "agency": True,
        "station_type": True,
        "unmet_need_index": ":.3f",
        "pct_no_vehicle": ":.1f",
        "pct_nonwhite": ":.1f",
        "total_amenities": True,
    },
    labels={
        "pct_no_vehicle": "% zero-vehicle households",
        "pct_nonwhite": "% non-white residents",
        "total_amenities": "Total amenities",
        "unmet_need_index": "Unmet need index",
        "station_type": "Type",
    },
    height=600,
    title="Station equity in three dimensions",
)
fig_3d.update_layout(
    scene=dict(
        xaxis_title="% zero-vehicle households",
        yaxis_title="% non-white residents",
        zaxis_title="Total amenities",
        bgcolor="white",
    ),
    paper_bgcolor="white",
    coloraxis_colorbar=dict(
        title="Unmet need",
        thickness=14,
        len=0.5,
    ),
    margin=dict(l=0, r=0, t=50, b=0),
)
display(HTML(fig_3d.to_html(full_html=False, include_plotlyjs="cdn")))